# 04 — Benchmarks reproductibles du lac V2

Chaque comparaison répète la même agrégation matérialisée trois fois et vérifie une empreinte du résultat. Une valeur plus rapide mais différente est rejetée.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import polars as pl

from poly_data.analysis.bench import repeat_benchmark
from poly_data.benchmark import benchmark_source
from poly_data.io.parquet_store import ParquetStore
from poly_data.notebooks import resolve_notebook_context, source_inventory

ctx = resolve_notebook_context()
store = ParquetStore(ctx.root)
SOURCES = ['order_filled_v2', 'trades', 'market_outcomes']
print({'root': str(ctx.root), 'mode': ctx.mode, 'revision': ctx.revision, 'polars': pl.__version__, 'duckdb': duckdb.__version__})
source_inventory(store, SOURCES)

## Coût de lecture des sources

Les tailles, fichiers et temps de scan sont rapportés avant toute comparaison d'agrégation.

In [ ]:
pl.DataFrame([{'source': source, **benchmark_source(store, source)} for source in SOURCES])

## Même agrégation froide, mêmes résultats

Les deux moteurs lisent les Parquet de ``trades`` et calculent le volume et le nombre de transactions par marché et issue.

In [ ]:
TRADES_GLOB = (ctx.root / 'trades' / '**' / '*.parquet').as_posix()

def polars_summary():
    return (
        store.scan('trades').group_by(['market_id', 'nonusdc_side'])
        .agg([pl.len().cast(pl.Int64).alias('n_trades'), pl.col('usd_amount').sum().round(8).cast(pl.Float64).alias('usd_volume')])
        .sort(['market_id', 'nonusdc_side']).collect()
    )

def duckdb_summary():
    query = '''
        SELECT market_id, nonusdc_side, CAST(COUNT(*) AS BIGINT) AS n_trades,
               CAST(ROUND(SUM(usd_amount), 8) AS DOUBLE) AS usd_volume
        FROM read_parquet(?)
        GROUP BY market_id, nonusdc_side
        ORDER BY market_id, nonusdc_side
    '''
    return pl.from_arrow(duckdb.execute(query, [TRADES_GLOB]).arrow())


In [ ]:
polars_cold = repeat_benchmark('cold_end_to_end', 3, polars_summary).with_columns(pl.lit('polars').alias('engine'))
duckdb_cold = repeat_benchmark('cold_end_to_end', 3, duckdb_summary).with_columns(pl.lit('duckdb').alias('engine'))
assert polars_cold['result_sha256'].n_unique() == duckdb_cold['result_sha256'].n_unique() == 1
assert polars_cold['result_sha256'][0] == duckdb_cold['result_sha256'][0]
cold = pl.concat([polars_cold, duckdb_cold])
cold

## Slices matérialisés (mesure séparée)

Cette mesure ne lit pas le lac : elle mesure seulement un filtre sur un résumé déjà calculé. Elle est volontairement présentée séparément du benchmark bout-en-bout.

In [ ]:
polars_cached_frame = polars_summary()
duckdb_cached_frame = duckdb_summary()
polars_cached = repeat_benchmark('cached_summary_slice', 3, lambda: polars_cached_frame.filter(pl.col('usd_volume') > 0)).with_columns(pl.lit('polars').alias('engine'))
duckdb_cached = repeat_benchmark('cached_summary_slice', 3, lambda: duckdb_cached_frame.filter(pl.col('usd_volume') > 0)).with_columns(pl.lit('duckdb').alias('engine'))
assert polars_cached['result_sha256'][0] == duckdb_cached['result_sha256'][0]
cached = pl.concat([polars_cached, duckdb_cached])
cached

In [ ]:
summary = pl.concat([cold, cached]).group_by(['label', 'engine']).agg([
    pl.col('seconds').min().alias('min_seconds'),
    pl.col('seconds').median().alias('median_seconds'),
    pl.col('seconds').max().alias('max_seconds'),
    pl.col('peak_rss_mb').median().alias('median_peak_rss_mb'),
    pl.col('rows_out').first(),
]).sort(['label', 'engine'])
summary

In [ ]:
plot = summary.to_pandas()
ax = plot.pivot(index='label', columns='engine', values='median_seconds').plot.bar(figsize=(8, 4), ylabel='median seconds')
ax.set_title('Cold and cached workloads are reported separately')
plt.tight_layout()